In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, BooleanType
from datetime import datetime

dbutils.widgets.text("bucket", "DataStagingBucket", "S3 Bucket")
dbutils.widgets.text("env", "prod", "Environment")

# 1. Define the path to CSV file
s3_bucket = dbutils.widgets.get("bucket")
env = dbutils.widgets.get("env")

# -------------------- csv path, target table name and schema -------------------
file_path = f"s3a://{s3_bucket}/edw_exports/{env}/requestidsince2022-05__20260309.csv"
target_table_name = "default.factrequestdetails"
new_table_name = "default.factmissingaxisrequests"

# schema mapping CSV's columns
csv_schema = StructType([
    StructField("vcVisibleRequestID", StringType(), True)
])

# 2. Read the CSV into a DataFrame
csv_df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("ignoreTrailingWhiteSpace", "true")
    .schema(csv_schema)
    .load(file_path)
)

# 3. Delta table
delta_df = (spark.read.table(target_table_name)
    .filter("startdate >= '2022-05-01'")
    .select("visualrequestfilenumber")
    .distinct()
)

# 4. left_anti join
missing_ids_df = delta_df.join(
    csv_df,
    delta_df["visualrequestfilenumber"] == csv_df["vcVisibleRequestID"],
    how="left_anti"
)

# 3. View the results
display(missing_ids_df)

print(f"Number of missing IDs: {missing_ids_df.count()}")

# 4. Save miss requests into the new Delta table
(missing_ids_df.write
    .format("delta")
    .mode("overwrite") # Overwrites the table if it already exists
    .saveAsTable(new_table_name)
)